# 02_gold
Goal: build the business tables that answer the main question:
**where do we lose our customers, before they buy or after their first order?**

- **Input:** `dev.ecommerce.events_silver` · 233,080,143 rows · one row = one event (view / cart / purchase)
- **Language:** SQL (`CREATE OR REPLACE TABLE ... AS SELECT`). Safe to run again: tables are rebuilt, never appended.
- **How to read:** each table has a short note, the `CREATE` cell, then `-- check:` cells that prove the numbers in the note.
- **Findings are not here.** This notebook only builds and checks tables. Findings are in `03_analysis`.

## Tables
| Table | One row is | Rows | Status |
|---|---|---|---|
| `gold_orders` | one order | 3,063,118 | ✅ done |
| `gold_funnel` | one month (Dec 2019, Jan 2020) | 2 | ✅ done |
| `gold_funnel_category` | one month + one main category | 28 | ✅ done · ⚠️ labels not reliable |
| `gold_funnel_brand` | one month + one brand | 9,724 | ✅ done |
| `gold_funnel_price` | one month + one price band | 12 | ✅ done |
| cart abandonment (per session) | | | ⬜ next |
| overview | | | ⬜ |
| retention | | | ⬜ |

In [0]:
%sql
USE CATALOG dev;
USE SCHEMA ecommerce

## gold_orders
An order = one user's purchases in one session.
`order_id` = user_session + '_' + user_id, because one session had 2 users.
- Columns: `order_id`, `user_id`, `order_time` (first purchase time), `item_count` (one purchase row = one item), `order_value`
- 3,063,118 orders · `order_id` is unique · 0 purchases with a NULL price
- 1,203 orders have 10+ items (0.04% of orders, 0.58% of revenue). Maybe resellers. Kept, because the effect is small.

In [0]:
%sql
create or replace table dev.ecommerce.gold_orders as 
select 
concat(user_session, '_', user_id) as order_id, 
user_id as user_id,
min(event_time) as order_time,
count(*) as item_count,
sum(price) as order_value 
from dev.ecommerce.events_silver
where event_type = 'purchase' 
group by  
user_session, user_id 
 

In [0]:
%sql
-- check: order_id is unique (both numbers must be equal: 3,063,118)
SELECT count(DISTINCT order_id) AS distinct_orders, count(*) AS rows
FROM dev.ecommerce.gold_orders

In [0]:
%sql
-- check: orders with 10+ items, share of orders and revenue (0.04% and 0.58%)
SELECT
  count_if(item_count >= 10)                                                    AS big_orders,
  round(count_if(item_count >= 10) / count(*) * 100, 2)                         AS big_orders_pct,
  round(sum(CASE WHEN item_count >= 10 THEN order_value END) / sum(order_value) * 100, 2) AS big_revenue_pct
FROM dev.ecommerce.gold_orders

## gold_funnel
The funnel counts **users**, not sessions or visits: a user is counted once per month (`count(DISTINCT user_id)`).
- **December and January only.** Before 8 Nov, cart tracking was not reliable: in October more people bought (347,118)
  than added to cart (337,117), and on 8 Nov carts jumped from 18,908 to 74,953 in one day while purchases stayed the same.

| Column | Meaning |
|---|---|
| `viewers` / `carters` / `buyers` | users who viewed / added to cart / bought in that month |
| `lost_at_cart` | viewers - carters |
| `lost_at_purchase` | carters - buyers |
| `cart_rate_pct` | carters / viewers · one step: view → cart |
| `cart_to_purchase_pct` | buyers / carters · one step: cart → purchase |
| `purchase_rate_pct` | buyers / viewers · the whole way: view → purchase |

- **Checks:** one row per month · viewers ≥ carters ≥ buyers · lost columns add up · all rates between 0 and 100
- **Limits:** about 1% of buyers have no cart event · the data ends on 31 Jan

In [0]:
%sql
-- check: cart tracking before 8 Nov (carts jump on 8 Nov, purchases stay flat)
SELECT event_date,
       count_if(event_type = 'cart')     AS carts,
       count_if(event_type = 'purchase') AS purchases
FROM dev.ecommerce.events_silver
WHERE event_date BETWEEN '2019-11-01' AND '2019-11-20'
GROUP BY event_date
ORDER BY event_date

In [0]:
%sql

create or replace table dev.ecommerce.gold_funnel as 
with x as (
    select date_format(event_date, 'yyyy-MM')                                    as month,
count(distinct case when event_type = 'view'     then user_id end)   as viewers,
count(distinct case when event_type = 'cart'     then user_id end)   as carters,
count(distinct case when event_type = 'purchase' then user_id end)   as buyers
from dev.ecommerce.events_silver
where event_date >= '2019-12-01'
group by month
)

select month,
viewers,
carters,
buyers,
viewers - carters                  as lost_at_cart,
carters - buyers                   as lost_at_purchase,
round(carters / viewers * 100, 1)  as cart_rate_pct,
round(buyers  / viewers * 100, 1)  as purchase_rate_pct,
round(buyers  / carters *100, 1)   as cart_to_purchase_pct
from x 
order by month

In [0]:
%sql
-- check: one row per month, viewers >= carters >= buyers, rates between 0 and 100
SELECT * FROM dev.ecommerce.gold_funnel ORDER BY month

## gold_funnel_category
The same funnel as `gold_funnel`, split by **main category** (the part of `category_code` before the first dot, `split_part(category_code, '.', 1)`).
- **One row = one month + one main category.** December and January only, same reason as above.
- Same columns and formulas as `gold_funnel`.
- A user who views products in 3 categories is counted in all 3 rows, so the category rows do **not** add up to the `gold_funnel` total.
- `unknown` (about 20% of events) is kept as its own row.
- Small categories (`medicine`, `stationery`, `country_yard`) are kept. Their rates still rest on hundreds of buyers. Check `viewers` before reading a rate.
- **Checks:** viewers ≥ carters ≥ buyers · all rates between 0 and 100

- ⚠️ **Category labels are not reliable.** About 46% of Apple events and 40% of Samsung events have a category the brand doesn't sell (e.g. Apple phones priced 443 to 1,597 labelled `construction.tools.light`). So findings and charts are **not** based on category. Brand and price are used instead.

In [0]:
%sql
CREATE OR REPLACE TABLE dev.ecommerce.gold_funnel_category AS
WITH x AS (
  SELECT
    date_format(event_date, 'yyyy-MM')     AS month,
    split_part(category_code, '.', 1)      AS main_category,
    count(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS viewers,
    count(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS carters,
    count(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
  FROM dev.ecommerce.events_silver
  WHERE event_date >= '2019-12-01'          -- Dec + Jan only (cart tracking)
  GROUP BY month, main_category
)
SELECT
  month,
  main_category,
  viewers,
  carters,
  buyers,
  viewers - carters                      AS lost_at_cart,
  carters - buyers                       AS lost_at_purchase,
  round(carters / viewers * 100, 1)      AS cart_rate_pct,
  round(buyers  / carters * 100, 1)      AS cart_to_purchase_pct,
  round(buyers  / viewers * 100, 1)      AS purchase_rate_pct
FROM x

In [0]:
%sql
-- check: 28 rows (14 main categories x 2 months)
SELECT * FROM dev.ecommerce.gold_funnel_category ORDER BY month, viewers DESC

In [0]:
%sql
-- check: category labels per brand (Apple and Samsung show up in construction, sport, ...)
SELECT brand, split_part(category_code, '.', 1) AS main_category, count(*) AS events
FROM dev.ecommerce.events_silver
WHERE brand IN ('apple', 'samsung')
GROUP BY brand, main_category
ORDER BY brand, events DESC

In [0]:
%sql
-- check: Apple products labelled construction are phone-priced (443 to 1,597)
SELECT category_code, product_id, round(avg(price), 0) AS avg_price, count(*) AS events
FROM dev.ecommerce.events_silver
WHERE brand = 'apple' AND category_code LIKE 'construction%'
GROUP BY category_code, product_id
ORDER BY events DESC
LIMIT 10

## gold_funnel_brand
The same funnel as `gold_funnel`, split by **brand**.
- **One row = one month + one brand.** December and January only.
- Same columns and formulas. Rates use `nullif(..., 0)`, because some tiny brands have 0 carters.
- 5,325 brands. About half have fewer than 1,000 events, so their rates are not reliable. **Read rates only for big brands** (sort by `viewers`).
- `unknown` brand is kept as its own row.
- **Brand check:** only 1% of products (2,132 of 214,066) have more than one brand, so brand labels are reliable (unlike category).

In [0]:
%sql
CREATE OR REPLACE TABLE dev.ecommerce.gold_funnel_brand AS
WITH x AS (
  SELECT
    date_format(event_date, 'yyyy-MM') AS month,
    brand,
    count(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS viewers,
    count(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS carters,
    count(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
  FROM dev.ecommerce.events_silver
  WHERE event_date >= '2019-12-01'
  GROUP BY month, brand
)
SELECT
  month,
  brand,
  viewers,
  carters,
  buyers,
  viewers - carters                              AS lost_at_cart,
  carters - buyers                               AS lost_at_purchase,
  round(carters / nullif(viewers, 0) * 100, 1)   AS cart_rate_pct,
  round(buyers  / nullif(carters, 0) * 100, 1)   AS cart_to_purchase_pct,
  round(buyers  / nullif(viewers, 0) * 100, 1)   AS purchase_rate_pct
FROM x

In [0]:
%sql
-- check: 9,724 rows; read rates only for big brands
SELECT * FROM dev.ecommerce.gold_funnel_brand
WHERE month = '2019-12'
ORDER BY viewers DESC
LIMIT 20

In [0]:
%sql
-- check: brand labels are reliable (2,132 of 214,066 products have more than one brand, about 1%)
WITH p AS (
  SELECT product_id, count(DISTINCT brand) AS brands
  FROM dev.ecommerce.events_silver
  WHERE brand <> 'unknown'
  GROUP BY product_id
)
SELECT count_if(brands > 1) AS products_with_many_brands,
       count(*)             AS products,
       round(count_if(brands > 1) / count(*) * 100, 1) AS pct
FROM p

## gold_funnel_price
The same funnel as `gold_funnel`, split by **price band**. Question: does price change *where* customers give up?
- **One row = one month + one price band.** December and January only, 12 rows (2 months x 6 bands).
- **Bands come from percentiles**, rounded: 25% of events are under 65, 50% under 164, 75% under 349, 90% under 741, 95% under 989 (max 2,574). The top quarter (349 to 2,574) was too wide for one band, so it is split in three.
- First band has no lower edge, last band no upper edge, so no price falls outside.
- Events with a NULL price are left out (0.2% of events, none is a purchase).
- Band names start with a number (`1: <65`) so they sort by price in SQL and Tableau.
- A user can be counted in several bands, so `viewers` can't be summed across bands. Rates are fine (each band on its own).

In [0]:
%sql
-- check: band edges come from percentiles (64.96, 164.06, 349.04, 741.13, 989.47; min 0.77, max 2,574.07)
SELECT percentile_approx(price, array(0.25, 0.5, 0.75, 0.9, 0.95)) AS edges,
       min(price) AS min,
       max(price) AS max
FROM dev.ecommerce.events_silver

In [0]:
%sql
CREATE OR REPLACE TABLE dev.ecommerce.gold_funnel_price AS
WITH x AS (
  SELECT
    date_format(event_date, 'yyyy-MM') AS month,
    CASE
      WHEN price < 65   THEN '1: <65'
      WHEN price < 165  THEN '2: 65-165'
      WHEN price < 350  THEN '3: 165-350'
      WHEN price < 750  THEN '4: 350-750'
      WHEN price < 1000 THEN '5: 750-1000'
      ELSE                   '6: 1000+'
    END AS price_band,
    count(DISTINCT CASE WHEN event_type = 'view'     THEN user_id END) AS viewers,
    count(DISTINCT CASE WHEN event_type = 'cart'     THEN user_id END) AS carters,
    count(DISTINCT CASE WHEN event_type = 'purchase' THEN user_id END) AS buyers
  FROM dev.ecommerce.events_silver
  WHERE event_date >= '2019-12-01'
    AND price IS NOT NULL
  GROUP BY month, price_band
)
SELECT
  month,
  price_band,
  round(buyers  / nullif(viewers, 0) * 100, 1)   AS purchase_rate_pct,
  viewers,
  carters,
  buyers,
  viewers - carters                              AS lost_at_cart,
  carters - buyers                               AS lost_at_purchase,
  round(carters / nullif(viewers, 0) * 100, 1)   AS cart_rate_pct,
  round(buyers  / nullif(carters, 0) * 100, 1)   AS cart_to_purchase_pct
FROM x

In [0]:
%sql
-- check: 12 rows (2 months x 6 bands), bands sort in price order
SELECT month, price_band, cart_rate_pct, cart_to_purchase_pct
FROM dev.ecommerce.gold_funnel_price
ORDER BY month, price_band